# 07 — Explainability and Error Analysis

**Objective** — inspect where the selected models fail and which features most influence predictive performance.

**Input** — final grouped train/test split and selected regression/classification models.

**Output** — error tables and permutation-importance summaries for both tasks.

## 1. Rebuild the Final Evaluation Split

In [ ]:
import pandas as pd
from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_regression_candidates, get_classification_candidates
from src.evaluation import regression_error_table, classification_error_table, permutation_importance_table
df = pd.read_csv('data/processed/analytical_dataset.csv')
X, y, groups = prepare_feature_frame(df)
Xtr, Xte, ytr, yte, gtr, gte = grouped_train_test_split(X, y, groups)
preprocessor = build_preprocessor(Xtr)

## 2. Regression Error Analysis

In [ ]:
reg = get_regression_candidates(preprocessor)['Random Forest'].fit(Xtr, ytr)
rpred = reg.predict(Xte)
regression_error_table(pd.DataFrame({'Customer ID': gte.values}), yte.reset_index(drop=True), rpred).head(10)

### Regression Permutation Importance

In [ ]:
permutation_importance_table(reg, Xte, yte, 'neg_mean_absolute_error', n_repeats=5).head(10)

## 3. Classification Error Analysis

In [ ]:
ytr_c, yte_c, threshold = derive_high_cost_labels(ytr, yte)
clf = get_classification_candidates(preprocessor)['HistGradientBoosting'].fit(Xtr, ytr_c)
score = clf.predict_proba(Xte)[:, 1]
pred = (score >= 0.5).astype(int)
classification_error_table(pd.DataFrame({'Customer ID': gte.values}), yte_c.reset_index(drop=True), pred, score).query("error_type != 'correct'").head(10)

### Classification Permutation Importance

In [ ]:
permutation_importance_table(clf, Xte, yte_c, 'average_precision', n_repeats=5).head(10)

## Takeaways

Error analysis makes model limitations visible instead of relying on aggregate metrics alone. Permutation importance describes how much predictive performance depends on each feature; it does not establish medical causality.